In [35]:
import pandas as pd
import requests
import statsmodels as sm
import numpy as np
import matplotlib as plt
import seaborn as sns
import urllib3
import csv
import time
from scipy import io
import importlib

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [36]:
DataFrame_incidenti = pd.read_csv('DataFrame_incidenti', na_values=['', 'NaN', 'null'], keep_default_na=False)
DataFrame_incidenti.isna().sum()

Unnamed: 0                 0
NOME_COMUNE                0
TIPO_COMUNE                0
CODICE_AREA                0
CODICE_REGIONE             0
ANNO                       0
SUPERFICIE_KMQ             0
POPOLAZIONE                0
RIPARTIZIONE_GEOGRAFICA    0
TIPO_INCIDENTE             0
RISULTATO_INCIDENTE        0
NUMERO_INCIDENTI           0
ZONA_ALTIMETRICA           0
ALTITUDINE_m               0
PRESENZA_MONTAGNE          0
COMUNE_LITORANEO           0
dtype: int64

# Raggruppp i dati separate le categorie e prendendo il massimo per popolazione e superficie

Raggruppo i dati per comune, anno e categoria per evitare doppioni, sommo il totale degli incidenti e prendo il valore massimo per la popolazione e la superficie territoriale.


In [47]:
df_intermedio = DataFrame_incidenti.groupby(
    ['NOME_COMUNE', 'ANNO', 'CODICE_REGIONE', 'RIPARTIZIONE_GEOGRAFICA', 'ZONA_ALTIMETRICA', 'ALTITUDINE_m', 'RISULTATO_INCIDENTE', 'PRESENZA_MONTAGNE', 'COMUNE_LITORANEO']
).agg({
    'NUMERO_INCIDENTI': 'sum',
    'POPOLAZIONE': 'max',
    'SUPERFICIE_KMQ': 'max'
}).reset_index()
df_intermedio

,NOME_COMUNE,ANNO,CODICE_REGIONE,RIPARTIZIONE_GEOGRAFICA,ZONA_ALTIMETRICA,ALTITUDINE_m,RISULTATO_INCIDENTE,PRESENZA_MONTAGNE,COMUNE_LITORANEO,NUMERO_INCIDENTI,POPOLAZIONE,SUPERFICIE_KMQ
0,Abano Terme,2010,VENETO,Nord-est,pianura,14.0,EVENTI,NON MONTANO,NON LITORANEO,80,19349.0,21.41
1,Abano Terme,2010,VENETO,Nord-est,pianura,14.0,FERITI,NON MONTANO,NON LITORANEO,101,19349.0,21.41
2,Abano Terme,2010,VENETO,Nord-est,pianura,14.0,MORTI,NON MONTANO,NON LITORANEO,0,19349.0,21.41
3,Abano Terme,2011,VENETO,Nord-est,pianura,14.0,EVENTI,NON MONTANO,NON LITORANEO,60,19349.0,21.41
4,Abano Terme,2011,VENETO,Nord-est,pianura,14.0,FERITI,NON MONTANO,NON LITORANEO,74,19349.0,21.41
...,...,...,...,...,...,...,...,...,...,...,...,...
339112,Zungri,2023,CALABRIA,Sud,collina interna,554.0,FERITI,NON MONTANO,NON LITORANEO,1,1985.0,23.46
339113,Zungri,2023,CALABRIA,Sud,collina interna,554.0,MORTI,NON MONTANO,NON LITORANEO,0,1985.0,23.46
339114,Zungri,2024,CALABRIA,Sud,collina interna,554.0,EVENTI,NON MONTANO,NON LITORANEO,3,1985.0,23.46
339115,Zungri,2024,CALABRIA,Sud,collina interna,554.0,FERITI,NON MONTANO,NON LITORANEO,5,1985.0,23.46


Trasformiamo le righe in colonne affiancate (Eventi, Feriti, Morti)


In [38]:
incidenti_agg = df_intermedio.pivot_table(
    index=['NOME_COMUNE', 'ANNO', 'CODICE_REGIONE', 'RIPARTIZIONE_GEOGRAFICA', 'ZONA_ALTIMETRICA', 'POPOLAZIONE', 'SUPERFICIE_KMQ'],
    columns='RISULTATO_INCIDENTE',
    values='NUMERO_INCIDENTI',
    aggfunc='sum'
).reset_index()

In [39]:
# Calcoliamo i nostri indicatori chiave sul totale degli eventi
incidenti_agg['DENSITA_INCIDENTI'] = incidenti_agg['EVENTI'] / incidenti_agg['SUPERFICIE_KMQ']
incidenti_agg['TASSO_SU_POPOLAZIONE'] = (incidenti_agg['EVENTI'] / incidenti_agg['POPOLAZIONE']) * 1000

# Controllo finale
incidenti_agg

RISULTATO_INCIDENTE,NOME_COMUNE,ANNO,CODICE_REGIONE,RIPARTIZIONE_GEOGRAFICA,ZONA_ALTIMETRICA,POPOLAZIONE,SUPERFICIE_KMQ,EVENTI,FERITI,MORTI,DENSITA_INCIDENTI,TASSO_SU_POPOLAZIONE
0,Abano Terme,2010,VENETO,Nord-est,pianura,19349.0,21.41,80,101,0,3.736572,4.134581
1,Abano Terme,2011,VENETO,Nord-est,pianura,19349.0,21.41,60,74,1,2.802429,3.100935
2,Abano Terme,2012,VENETO,Nord-est,pianura,19349.0,21.41,30,36,0,1.401214,1.550468
3,Abano Terme,2013,VENETO,Nord-est,pianura,19349.0,21.41,64,82,2,2.989257,3.307664
4,Abano Terme,2014,VENETO,Nord-est,pianura,19349.0,21.41,67,80,1,3.129379,3.462711
...,...,...,...,...,...,...,...,...,...,...,...,...
113034,Zungri,2020,CALABRIA,Sud,collina interna,1985.0,23.46,1,2,0,0.042626,0.503778
113035,Zungri,2021,CALABRIA,Sud,collina interna,1985.0,23.46,1,1,0,0.042626,0.503778
113036,Zungri,2022,CALABRIA,Sud,collina interna,1985.0,23.46,1,2,0,0.042626,0.503778
113037,Zungri,2023,CALABRIA,Sud,collina interna,1985.0,23.46,1,1,0,0.042626,0.503778


In [40]:
# qui semplicemente elimino il nome della colonna "RISULTATO_INCIDENTE" che si può vedere come prima colonna, è causato dal pivot del dataframe
incidenti_agg.columns.name = None


In [41]:
incidenti_agg

,NOME_COMUNE,ANNO,CODICE_REGIONE,RIPARTIZIONE_GEOGRAFICA,ZONA_ALTIMETRICA,POPOLAZIONE,SUPERFICIE_KMQ,EVENTI,FERITI,MORTI,DENSITA_INCIDENTI,TASSO_SU_POPOLAZIONE
0,Abano Terme,2010,VENETO,Nord-est,pianura,19349.0,21.41,80,101,0,3.736572,4.134581
1,Abano Terme,2011,VENETO,Nord-est,pianura,19349.0,21.41,60,74,1,2.802429,3.100935
2,Abano Terme,2012,VENETO,Nord-est,pianura,19349.0,21.41,30,36,0,1.401214,1.550468
3,Abano Terme,2013,VENETO,Nord-est,pianura,19349.0,21.41,64,82,2,2.989257,3.307664
4,Abano Terme,2014,VENETO,Nord-est,pianura,19349.0,21.41,67,80,1,3.129379,3.462711
...,...,...,...,...,...,...,...,...,...,...,...,...
113034,Zungri,2020,CALABRIA,Sud,collina interna,1985.0,23.46,1,2,0,0.042626,0.503778
113035,Zungri,2021,CALABRIA,Sud,collina interna,1985.0,23.46,1,1,0,0.042626,0.503778
113036,Zungri,2022,CALABRIA,Sud,collina interna,1985.0,23.46,1,2,0,0.042626,0.503778
113037,Zungri,2023,CALABRIA,Sud,collina interna,1985.0,23.46,1,1,0,0.042626,0.503778
